Example
- LRU Cache
- Parking Lot
- Elevator
- ATM
- Library Management
- Meeting Room Scheduler
- Rate Limiter
- File System
- Autocomplete
- Vending Machine
- Task Scheduler
- Snake Game
- Spreadsheet

Polymorphism, encapsulation, inheritance 



In [5]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional
import uuid

class Status(Enum):
    ACTIVE = "active"
    INACTIVE = "inactive"

@dataclass
class Entity:
    id: str
    name: str
    status: Status = Status.ACTIVE

class EntityManager:
    def __init__(self) -> None:
        self.entities: Dict[str, Entity] = {}

    def create(self, name: str) -> Entity:
        entity = Entity(id=str(uuid.uuid4()), name=name)
        self.entities[entity.id] = entity
        return entity

    def get(self, entity_id: str) -> Optional[Entity]:
        return self.entities.get(entity_id)

    def delete(self, entity_id: str) -> bool:
        if entity_id not in self.entities:
            return False
        del self.entities[entity_id]
        return True

    def list_all(self) -> List[Entity]:
        return list(self.entities.values())

    def update_status(self, entity_id: str, status: Status) -> bool:
        entity = self.entities.get(entity_id)
        if not entity:
            return False
        entity.status = status
        return True


## LRU Cache

LRU = Least Recently Used

You need a cache with:
- `get(key)` -> return value if present, ele `-1`
- `put(key, value)` -> insert/update
- when capacity is full, evict the least recently used item 

Core idea:
- use a hash map: `key -> node` for O(1) lookup
- a doubly linked list to maintain usage order

Order:
- most recently used near the front
- least recently used near the back

When:
- `get(key)` -> move node to front
- `put(key, value)`:
    - if exists -> update + move to front
    - if new and full -> remove from back
    - insert new node at front



In [12]:
class Node:
    def __init__(self, key: int, value: int):
        self.key = key 
        self.value = value 
        self.prev = None
        self.next = None 

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity 
        self.cache = {}

        self.head = Node(0, 0)
        self.tail = Node(0, 0)
        self.head.next = self.tail 
        self.tail.prev = self.head

    def _remove(self, node: Node) -> None:
        prev_node = node.prev 
        next_node = node.next 
        prev_node.next = next_node 
        next_node.prev = prev_node 

    def _add_to_front(self, node:Node) -> None:
        first_real = self.head.next 
        self.head.next = node
        node.prev = self.head 
        node.next = first_real 
        first_real.prev = node 

    def _move_to_front(self, node:Node) -> None:
        self._remove(node)
        self._add_to_front(node)

    def _pop_lru(self) -> int:
        lru = self.tail.prev 
        self._remove(lru)
        return lru 

    def get(self, key:int) -> int:
        if key not in self.cache:
            return -1 
        node = self.cache[key]
        self._move_to_front(node)
        return node.value 

    def put(self, key:int, value:int) -> None:
        if key in self.cache:
            node = self.cache[key]
            node.value = value 
            self._move_to_front(node)
            return 
        if len(self.cache) == self.capacity: 
            lru = self._pop_lru()
            del self.cache[lru.key]

        new_node = Node(key, value) 
        self.cache[key] = new_node 
        self._add_to_front(new_node) 

In [13]:
cache = LRUCache(2)
cache.put(1, 10)   # {1=10}
cache.put(2, 20)   # {2=20, 1=10}
cache.get(1)       # returns 10, order becomes {1=10, 2=20}
cache.put(3, 30)   # evict key 2, cache becomes {3=30, 1=10}
cache.get(2)       # returns -1
cache.put(4, 40)   # evict key 1, cache becomes {4=40, 3=30}
cache.get(1)       # -1
cache.get(3)       # 30
cache.get(4)       # 40

40

## Parking Lot

In [14]:
from enum import Enum
from dataclasses import dataclass, field


class VehicleType(Enum):
    MORTORCYCLE = "mortocycle"
    CAR = "car"
    TRUCK = "truck"

class SpotType(Enum):
    MOTORCYCLE = "motorcycle"
    COMPACT = "compact"
    LARGE = "large"


@dataclass 
class Vehicle:
    plate: str
    vehicle_type: VehicleType 

@dataclass 
class Spot:
    spot_id: str 
    spot_type: SpotType 
    occupant: Optional[str] = None

In [16]:
# class ParkingLot: 

## Order Book

- add order
- cancel order
- best bid / best ask
- simple matching
- partial fills


In [17]:
from collections import defaultdict, deque
from dataclasses import dataclass
from typing import Optional


@dataclass
class Order:
    order_id: str
    side: str   # "BUY" or "SELL"
    price: float
    quantity: int


class OrderBook:
    def __init__(self):
        self.buy_levels = defaultdict(deque)
        self.sell_levels = defaultdict(deque)

        # order_id -> order
        self.orders = {}

    def add_order(self, order):
        self.orders[order.order_id] = order 
            
        if order.side == "BUY":
            self.buy_levels[order.price].append(order)
        elif order.side == "SELL":
            self.sell_levels[order.price].append(order)
        else:
            raise ValueError("side must be BUY or SELL")



    def cancel_order(self, order_id: str) -> bool:
        if order_id not in self.orders:
            return False

        order = self.orders.pop(order_id)
        levels = self.buy_levels if order.side == "BUY" else self.sell_levels
        queue = levels[order.price]

        # remove from deque
        new_queue = deque(o for o in queue if o.order_id != order_id)
        if new_queue:
            levels[order.price] = new_queue
        else:
            del levels[order.price]

        return True


    def get_best_bid(self) -> Optional[float]:
        return max(self.buy_levels.keys()) if self.buy_levels else None

    def get_best_ask(self) -> Optional[float]:
        return min(self.sell_levels.keys()) if self.sell_levels else None

    def match(self) -> list[tuple[str, str, float, int]]:
        """
        Returns list of trades:
        (buy_order_id, sell_order_id, trade_price, trade_quantity)
        """
        trades = []

        while self.buy_levels and self.sell_levels:
            best_bid = self.get_best_bid()
            best_ask = self.get_best_ask()

            if best_bid < best_ask:
                break  # no crossing market

            buy_queue = self.buy_levels[best_bid]
            sell_queue = self.sell_levels[best_ask]

            buy_order = buy_queue[0]
            sell_order = sell_queue[0]

            trade_qty = min(buy_order.quantity, sell_order.quantity)
            trade_price = sell_order.price  # simple convention

            trades.append(
                (buy_order.order_id, sell_order.order_id, trade_price, trade_qty)
            )

            buy_order.quantity -= trade_qty
            sell_order.quantity -= trade_qty

            if buy_order.quantity == 0:
                buy_queue.popleft()
                del self.orders[buy_order.order_id]
                if not buy_queue:
                    del self.buy_levels[best_bid]

            if sell_order.quantity == 0:
                sell_queue.popleft()
                del self.orders[sell_order.order_id]
                if not sell_queue:
                    del self.sell_levels[best_ask]

        return trades

## Key-Value Store


In [18]:
import time
from dataclasses import dataclass
from typing import Any, Optional


@dataclass
class ValueEntry:
    value: Any
    expire_at: Optional[float] = None


class KeyValueStore:
    def __init__(self):
        self.store = {}

    def put(self, key: str, value: Any, ttl_seconds: Optional[int] = None) -> None:
        expire_at = time.time() + ttl_seconds if ttl_seconds is not None else None
        self.store[key] = ValueEntry(value=value, expire_at=expire_at)

    def get(self, key: str) -> Any:
        if key not in self.store:
            return None

        entry = self.store[key]
        if self._is_expired(entry):
            del self.store[key]
            return None

        return entry.value

    def delete(self, key: str) -> bool:
        if key not in self.store:
            return False
        del self.store[key]
        return True

    def exists(self, key: str) -> bool:
        if key not in self.store:
            return False

        entry = self.store[key]
        if self._is_expired(entry):
            del self.store[key]
            return False

        return True

    def _is_expired(self, entry: ValueEntry) -> bool:
        return entry.expire_at is not None and time.time() >= entry.expire_at

## Rate Limiter

In [20]:
import time
from collections import defaultdict, deque


class RateLimiter:
    def __init__(self, max_requests: int, window_seconds: int):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.requests = defaultdict(deque)  # user_id -> deque of timestamps

    def allow_request(self, user_id: str) -> bool:
        now = time.time()
        q = self.requests[user_id]

        # remove expired timestamps
        while q and q[0] <= now - self.window_seconds:
            q.popleft()

        if len(q) >= self.max_requests:
            return False

        q.append(now)
        return True